# LLM Sub-niche Validation

This notebook takes the clusters produced by `NearestNeighbors.ipynb` and uses an LLM to validate and refine them into **competitor sub-niches**.

**The problem:** embedding-based clustering groups apps that are semantically similar, but similarity ≠ direct competition. Two apps can use the same technology (e.g. AI image generation) yet target completely different user needs (portrait photos vs. logo design). The LLM is used to make this distinction.

**Pipeline:**
1. **Prompt design** — craft a system prompt that instructs the LLM to split each cluster into sub-niches of direct competitors
2. **Structured output** — use Pydantic models + OpenAI's `parse` API to guarantee a machine-readable JSON response
3. **LLM loop** — send each cluster to GPT-4o-mini and collect sub-niche groups
4. **Enrich DataFrame** — map sub-niche labels back to the original apps
5. **De-duplicate** — merge identical sub-niche names that appeared across different clusters
6. **Visualize** — UMAP plot coloured by sub-niche to verify spatial coherence

In [ ]:
import os
import json
import umap
import ast
import pandas as pd
import numpy as np
from openai import OpenAI
import plotly.express as px
from dotenv import load_dotenv

c:\Study\selfeducation\Universe_Group_test_task\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup

Load API credentials from `.env`. The OpenAI client is used for all LLM calls.

In [ ]:
load_dotenv(dotenv_path=".env")
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

## 2. Prompt & Response Schema Design

### System Prompt
The system prompt defines the LLM's role and the rules it must follow:
- **Direct competitors only** — apps that solve the same specific problem for the same user, not just apps in the same broad category
- **Exhaustive** — every app in the cluster must appear in exactly one sub-niche
- **Structured fields** — `sub_niche` (short name), `sub_niche_description` (2-3 sentences), `competitors` (list of app names)

### Pydantic Response Schema
Using OpenAI's structured output (`client.beta.chat.completions.parse`) with a Pydantic model guarantees the response always matches the expected shape — no manual JSON parsing or error-prone string extraction needed.

In [1]:
SYSTEM_PROMPT = """You are a mobile app market analyst. Analyze a group of apps from the same cluster and identify which ones are direct competitors.

Direct competitors share the same core use case and target the same user need — not just the same broad category.

For each sub-niche group provide:
- sub_niche: a short name for the sub-niche (e.g. "AI logo maker")
- sub_niche_description: 2-3 sentences describing the sub-niche — what problem it solves, who the target user is, and what distinguishes it from adjacent categories
- competitors: list of trackNames of apps that are direct competitors within that sub-niche

Rules:
- If all apps belong to one sub-niche, return a single-element list.
- If the cluster contains multiple distinct sub-niches, return one object per sub-niche.
- Every app in the cluster must appear in exactly one sub-niche (do not omit any app)."""


In [2]:
from pydantic import BaseModel

class SubNiche(BaseModel):
    sub_niche: str
    sub_niche_description: str
    competitors: list[str]

class ClusterAnalysis(BaseModel):
    sub_niches: list[SubNiche]


## 3. Data Loading

Load the clustered app dataset produced by `NearestNeighbors.ipynb`.  
The `cluster` column contains integer cluster IDs; `embedding_features` holds the pre-computed sentence embeddings (stored as strings in CSV — parsed back to arrays when needed).

In [7]:
df = pd.read_csv('clustered_apps_NN.csv')
df.head()

,trackName,description,overview,features,embedding_features,cluster
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,['AI-Powered Logo Generation with customizable...,"[0.026286382228136063, 0.014646739698946476, -...",0
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"['AI-generated logo design', 'Customizable let...","[0.029506200924515724, 0.018569650128483772, 0...",0
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"['AI-powered logo generation in seconds', 'Ove...","[0.02679363079369068, 0.02018563449382782, -0....",0
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"['Access to over 7000 logo templates', 'Divers...","[0.01727970503270626, 0.016116071492433548, 0....",1
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,['10K+ customizable logo templates across 13 c...,"[0.013803043402731419, 0.02349875494837761, 0....",1


## 4. LLM Prompting Functions

Two helper functions handle the LLM interaction:

- **`build_user_prompt`** — formats a cluster's apps into a readable text block. Each app gets its `trackName`, `overview`, and up to 10 `features`. Apps are separated by `---` so the model can clearly distinguish them. The `features` column comes back as a string from CSV, so `isinstance(features, list)` guards against both formats.

- **`analyze_cluster`** — sends the prompt to `gpt-4o-mini` using structured output (`response_format=ClusterAnalysis`). The parsed response is converted to a plain dict list via `.model_dump()` for easy JSON serialisation. Errors are caught and logged without stopping the loop.

In [ ]:
def build_user_prompt(cluster_df):
    """Format all apps in a cluster into a single prompt string for the LLM."""
    lines = []
    for _, row in cluster_df.iterrows():
        features = row["features"]
        # features is a list when in memory, but a string after CSV round-trip
        if isinstance(features, list):
            features_str = "; ".join(features[:10])   # cap at 10 to limit token usage
        else:
            features_str = str(features)[:300]         # truncate long strings

        lines.append(
            f"App: {row['trackName']}\n"
            f"Overview: {row['overview']}\n"
            f"Features: {features_str}\n"
        )
    return "\n---\n".join(lines)   # separator helps the model distinguish apps

In [ ]:
def analyze_cluster(cluster_id, cluster_df):
    """Send one cluster to the LLM and return a list of sub-niche dicts."""
    user_prompt = build_user_prompt(cluster_df)
    try:
        response = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.2,              # low temp for consistent, factual output
            response_format=ClusterAnalysis,  # enforces Pydantic schema on the response
        )
        result = response.choices[0].message.parsed   # already a ClusterAnalysis object
        return [sn.model_dump() for sn in result.sub_niches]  # convert to plain dicts
    except Exception as e:
        print(f"  Error for cluster {cluster_id}: {e}")
        return []   # skip failed clusters; don't abort the whole loop

## 5. Main Analysis Loop

Iterate over every valid cluster and call the LLM to split it into sub-niches.

**Scope:** currently limited to clusters 0–50 (the largest, most populated clusters) to control API cost during development. Change the filter to process all clusters.

Each result dict is extended with `cluster_id` so sub-niches can be traced back to their source cluster.

In [ ]:
all_results = []

# NaN-safe cast: cluster column may be float after CSV round-trip
df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce").fillna(-1).astype(int)

# Process clusters 0–50; extend to df["cluster"].unique() to cover all clusters
valid_clusters = sorted(df[df["cluster"] <= 50]["cluster"].unique().tolist())

for cluster_id in valid_clusters:
    cluster_df = df[df["cluster"] == cluster_id][["trackName", "overview", "features"]]
    print(f"Processing cluster {cluster_id} ({len(cluster_df)} apps)...")

    result = analyze_cluster(cluster_id, cluster_df)
    for item in result:
        item["cluster_id"] = int(cluster_id)  # int() converts numpy int64 → JSON-serialisable

    all_results.extend(result)

print(f"\nDone. Total sub-niches found: {len(all_results)}")
print(json.dumps(all_results[:3], indent=2, ensure_ascii=False))

## 6. Map Sub-niche Results Back to the DataFrame

For each sub-niche returned by the LLM:
1. Assign a temporary `sub_niche_id` (sequential index into `all_results`)
2. Build a `trackName → sub-niche fields` lookup
3. Left-join the lookup onto `df` — apps not covered by the LLM (unprocessed clusters) get `NaN`

In [11]:
# Assign a unique sub_niche_id to each sub-niche entry
for i, item in enumerate(all_results):
    item["sub_niche_id"] = i

# Build a mapping: trackName -> sub-niche fields
track_to_subniche = {}
for item in all_results:
    for track_name in item["competitors"]:
        track_to_subniche[track_name] = {
            "sub_niche_id": item["sub_niche_id"],
            "sub_niche": item["sub_niche"],
            "sub_niche_description": item["sub_niche_description"],
        }

df["sub_niche_id"] = df["trackName"].map(lambda x: track_to_subniche.get(x, {}).get("sub_niche_id"))
df["sub_niche"] = df["trackName"].map(lambda x: track_to_subniche.get(x, {}).get("sub_niche"))
df["sub_niche_description"] = df["trackName"].map(lambda x: track_to_subniche.get(x, {}).get("sub_niche_description"))

print(f"Matched: {df['sub_niche_id'].notna().sum()} / {len(df)} apps")
df[["trackName", "cluster", "sub_niche_id", "sub_niche", "sub_niche_description"]].head(10)


Matched: 361 / 4264 apps


,trackName,cluster,sub_niche_id,sub_niche,sub_niche_description
0,Logo Maker - AI Art Design,0,0.0,AI logo maker,This sub-niche focuses on applications that ut...
1,Arvin® - AI Logo Maker,0,0.0,AI logo maker,This sub-niche focuses on applications that ut...
2,AI Logo Generator & Design,0,0.0,AI logo maker,This sub-niche focuses on applications that ut...
3,Logo Maker _ AI Design Creator,1,2.0,Logo Maker,This sub-niche focuses on applications that al...
4,Logo Maker Shop: AI Creator,1,2.0,Logo Maker,This sub-niche focuses on applications that al...
5,Logo Maker: Art Design Creator,2,9.0,AI logo maker,This sub-niche focuses on applications that en...
6,Canva: AI Photo & Video Editor,3,12.0,Graphic Design & Content Creation,This sub-niche focuses on apps that enable use...
7,Rebrand - AI Logo Maker,0,0.0,AI logo maker,This sub-niche focuses on applications that ut...
8,AI Art Generator & Logo - Hexa,1,2.0,Logo Maker,This sub-niche focuses on applications that al...
9,AI Logo Generator - Mark,4,13.0,AI logo maker,This sub-niche focuses on applications that ut...


## 7. De-duplicate Sub-niche IDs

The LLM may independently assign the same sub-niche name to apps from different clusters (e.g. "AI logo maker" appearing in cluster 0 and cluster 2).  
This step consolidates them: build a `name → canonical_id` mapping from the unique sub-niche strings and re-map `sub_niche_id` so identical names share one integer id.

Result: a dense 0-based integer id per unique sub-niche name, stored as nullable `Int64` to preserve `NaN` for unmatched apps.

In [30]:
# Merge rows that share the same sub_niche name but got different sub_niche_ids
# across clusters. Assign a single canonical id (dense 0-based) per unique name.

sub_niche_to_canonical = {
    name: i
    for i, name in enumerate(df["sub_niche"].dropna().unique())
}

df["sub_niche_id"] = df["sub_niche"].map(sub_niche_to_canonical).astype("Int64")

n_before = df["sub_niche_id"].nunique()
print(f"Unique sub-niches after dedup: {n_before}  (was {df['sub_niche_id'].notna().sum()} rows mapped)")
df.head(10)

Unique sub-niches after dedup: 108  (was 361 rows mapped)


,trackName,description,overview,features,embedding_features,cluster,sub_niche_id,sub_niche,sub_niche_description,umap_x,umap_y
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,['AI-Powered Logo Generation with customizable...,"[0.026286382228136063, 0.014646739698946476, -...",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.159142,4.650215
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"['AI-generated logo design', 'Customizable let...","[0.029506200924515724, 0.018569650128483772, 0...",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.212003,4.594880
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"['AI-powered logo generation in seconds', 'Ove...","[0.02679363079369068, 0.02018563449382782, -0....",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.171481,4.649245
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"['Access to over 7000 logo templates', 'Divers...","[0.01727970503270626, 0.016116071492433548, 0....",1,1,Logo Maker,This sub-niche focuses on applications that al...,1.647011,4.401441
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,['10K+ customizable logo templates across 13 c...,"[0.013803043402731419, 0.02349875494837761, 0....",1,1,Logo Maker,This sub-niche focuses on applications that al...,1.673321,4.372460
5,Logo Maker: Art Design Creator,"Design stunning logos, flyers, and social medi...","Create stunning logos, flyers, and social medi...","['10,000+ ready-to-use templates crafted by pr...","[0.04362112283706665, -0.005856246221810579, -...",2,0,AI logo maker,This sub-niche focuses on applications that en...,0.549894,4.426378
6,Canva: AI Photo & Video Editor,Canva is your easy to use photo editor and vid...,Canva is a versatile graphic design app that c...,"['Create and customize social media graphics',...","[0.038620613515377045, 0.011229723691940308, 0...",3,2,Graphic Design & Content Creation,This sub-niche focuses on apps that enable use...,1.049313,4.895138
7,Rebrand - AI Logo Maker,Rebrand: Your AI-Powered Logo Design Studio\n\...,Rebrand is an AI-powered logo design studio th...,"['AI logo generator for unique designs', 'Cust...","[0.03344714641571045, 0.02044876106083393, -0....",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.151537,4.600802
8,AI Art Generator & Logo - Hexa,"Unlock your creativity with Hexa AI, the ultim...",Hexa AI is an innovative design app that harne...,['AI Generated Logos with customizable editing...,"[0.017201421782374382, 0.009349129162728786, 0...",1,1,Logo Maker,This sub-niche focuses on applications that al...,1.240843,4.531780
9,AI Logo Generator - Mark,Mark - AI Logo Generator\n\nNo design skills? ...,Mark is an AI-powered logo generator designed ...,"['Create custom letter and text logos', 'Choos...","[0.034139253199100494, 0.01772329956293106, 0....",4,0,AI logo maker,This sub-niche focuses on applications that ut...,1.686332,4.433528


## 8. Filter & Export

Keep only apps that were assigned a sub-niche (`df_niched`) and export a clean result table (`app_sub_niches.csv`) with renamed columns for readability.

In [31]:
df['sub_niche_id'].value_counts()

sub_niche_id
16     23
73     17
51     14
6      12
4      12
       ..
103     1
104     1
105     1
106     1
107     1
Name: count, Length: 108, dtype: Int64

In [34]:
df_niched = df[df["sub_niche_id"].notna()].reset_index(drop=True)
df_niched.head()

,trackName,description,overview,features,embedding_features,cluster,sub_niche_id,sub_niche,sub_niche_description,umap_x,umap_y
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,['AI-Powered Logo Generation with customizable...,"[0.026286382228136063, 0.014646739698946476, -...",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.159142,4.650215
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"['AI-generated logo design', 'Customizable let...","[0.029506200924515724, 0.018569650128483772, 0...",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.212003,4.594880
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"['AI-powered logo generation in seconds', 'Ove...","[0.02679363079369068, 0.02018563449382782, -0....",0,0,AI logo maker,This sub-niche focuses on applications that ut...,1.171481,4.649245
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"['Access to over 7000 logo templates', 'Divers...","[0.01727970503270626, 0.016116071492433548, 0....",1,1,Logo Maker,This sub-niche focuses on applications that al...,1.647011,4.401441
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,['10K+ customizable logo templates across 13 c...,"[0.013803043402731419, 0.02349875494837761, 0....",1,1,Logo Maker,This sub-niche focuses on applications that al...,1.673321,4.372460


In [35]:
df_result = df_niched[["trackName", "sub_niche_description", "sub_niche"]].rename(columns={
    "trackName": "AppName",
    "sub_niche_description": "Description",
    "sub_niche": "Sub-niche Name",
})

df_result.head()

,AppName,Description,Sub-niche Name
0,Logo Maker - AI Art Design,This sub-niche focuses on applications that ut...,AI logo maker
1,Arvin® - AI Logo Maker,This sub-niche focuses on applications that ut...,AI logo maker
2,AI Logo Generator & Design,This sub-niche focuses on applications that ut...,AI logo maker
3,Logo Maker _ AI Design Creator,This sub-niche focuses on applications that al...,Logo Maker
4,Logo Maker Shop: AI Creator,This sub-niche focuses on applications that al...,Logo Maker


In [38]:
df_result.to_csv("app_sub_niches.csv", index=False)

## 9. UMAP Visualization

Project the matched apps' embeddings to 2D using UMAP and colour by `sub_niche_id`.  
If the sub-niche assignments are correct, apps with the same sub-niche should cluster tightly together in the 2D projection — this serves as a visual sanity check.

> `embedding_features` was stored as a string in CSV, so `ast.literal_eval` parses it back to a Python list before converting to a numpy array.

In [ ]:
# embedding_features was saved as a string in CSV — parse back to float list
X = np.array(df_niched["embedding_features"].map(ast.literal_eval).tolist())

umap_model = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42,
    metric='cosine',   # consistent with how embeddings were trained
)
X_umap = umap_model.fit_transform(X)

df_niched = df_niched.copy()   # avoid SettingWithCopyWarning
df_niched['umap_x'] = X_umap[:, 0]
df_niched['umap_y'] = X_umap[:, 1]

In [ ]:
# Colour by sub_niche_id — apps sharing the same sub-niche should appear spatially close
fig = px.scatter(
    df_niched,
    x='umap_x',
    y='umap_y',
    color='sub_niche_id',
    hover_data=['trackName', 'sub_niche'],
    title="UMAP — apps coloured by sub-niche (LLM-validated)",
    width=1000,
    height=800,
    color_continuous_scale=px.colors.qualitative.Alphabet,
)
fig.show(renderer="browser")